# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
pip install evaluate

In [1]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets
from tqdm import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 46.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

### Data Preparation

In [3]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [ ]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [ ]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [ ]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)
model = model.to("cpu")
with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
batch_size = 128
model = model.to(device)

val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=batch_size, shuffle=False, collate_fn=transformers.default_data_collator,num_workers=4
)
acc = 0
total = 0
for batch in val_loader:
    input_ids=batch["input_ids"].to(device)
    attention_mask=batch["attention_mask"].to(device)
    token_type_ids=batch["token_type_ids"].to(device)
    true_labels = batch["labels"].to(device)
    with torch.no_grad():
        predicted = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        predicted = torch.argmax(predicted.logits,dim=1)

    acc += torch.sum(true_labels == predicted).to("cpu").numpy()
    total += len(true_labels)

accuracy = acc/total

In [ ]:
print("accuracy = ", accuracy)

accuracy =  0.9083848627256987


In [ ]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

Попробуем зафайнтьюнить "microsoft/deberta-v3-base"



In [24]:
model_name = "microsoft/deberta-v3-base"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2,
                                                                        id2label={0: "not_duplicate", 1: "duplicate"},
                                                                        label2id={"not_duplicate": 0, "duplicate": 1})

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [6]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [25]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

In [26]:
# import evaluate
import numpy as np
from sklearn.metrics import accuracy_score

# accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
    }

In [27]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./deberta-v3-qqp",
    learning_rate=2e-5,
    per_device_train_batch_size=48,
    per_device_eval_batch_size=48,
    num_train_epochs=0.25,
    #weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="epoch",
    eval_steps=500,
    # logging_dir="./logs",
    logging_steps=500,
    report_to=None,  # Disable wandb if not needed
    push_to_hub=False,  # Set to True if you want to push to Hugging Face Hub
)

In [28]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=qqp_preprocessed["train"],
    eval_dataset=qqp_preprocessed["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [22]:
import wandb

In [23]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

my_secret = user_secrets.get_secret("wandb_api_key") 

wandb.login(key=my_secret)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: eliyashev (eliyashev-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [29]:
trainer.train()


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy
500,0.330000,0.273425,0.884442


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=948, training_loss=0.3011662205563316, metrics={'train_runtime': 1589.7042, 'train_samples_per_second': 57.219, 'train_steps_per_second': 0.596, 'total_flos': 5986410088955904.0, 'train_loss': 0.3011662205563316, 'epoch': 0.2500659456607755})

In [30]:
# Evaluate the model
print("\nEvaluating model...")
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)


Evaluating model...


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Evaluation results: {'eval_loss': 0.25883471965789795, 'eval_accuracy': 0.8886965124907247, 'eval_runtime': 219.27, 'eval_samples_per_second': 184.385, 'eval_steps_per_second': 1.925, 'epoch': 0.2500659456607755}


### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

Реалзиация такая. Берем вопрос и пропускаем его вместе с полем text2 через токеназер и прогоняем это через весь набор данных. Затем подаём что получилось в модель, отбираем варианты с набольшей вероятностью дубликата и с вероятностью выше некотрого порога. Затем печатаем top-5 впоросов, или меньше если меньше прошло порог. Работает это всё не быстро.

In [31]:
MAX_LENGTH = 128

def get_qestion_pairs_tkn(question):
    def get_qestion_pairs(examples):
        result = tokenizer(
            question,
            examples["text2"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
        )

        return result

    return get_qestion_pairs

In [32]:
def find_similar(question,model,qqp,threshold = 0.8):
    qestion_pairs_tkn = get_qestion_pairs_tkn(question)
    qqp_preprocessed = qqp['validation'].map(qestion_pairs_tkn, batched=False)
    batch_size = 64
    model = model.to(device)

    val_loader = torch.utils.data.DataLoader(
        qqp_preprocessed, batch_size=batch_size, shuffle=False, collate_fn=transformers.default_data_collator,num_workers=1
    )
    pred_prob = np.array([])
    for batch in tqdm(val_loader):
        input_ids=batch["input_ids"].to(device)
        attention_mask=batch["attention_mask"].to(device)
        token_type_ids=batch["token_type_ids"].to(device)
        with torch.no_grad():
            predicted = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            pred = torch.softmax(predicted.logits, dim=1).data.cpu().numpy()
            pred = pred[:,1]
            pred_prob = np.append(pred_prob,pred)

    indx = np.argsort(pred_prob)[::-1]
    threshold = 0.8
    similar_questons = []
    for i in indx:
        if pred_prob[i]>threshold:
            similar_questons.append(qqp_preprocessed[i]['text2'])
        else:
            break
    return similar_questons

In [33]:
def print_similar_question(question):
    print(f"question is '{question}'")
    similar = find_similar(question,model,qqp,threshold = 0.8)
    if len(similar) == 0:
        print("no similar")
    else:
        print("similar questions:")
        for q in similar[:5]:
            print(q)

In [35]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [36]:
question = qqp['test'][0]['text1']
print_similar_question(question)

question is 'Would the idea of Trump and Putin in bed together scare you, given the geopolitical implications?'


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

  0%|          | 0/632 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 632/632 [05:55<00:00,  1.78it/s]

no similar


In [37]:
question = qqp['test'][1]['text1']
print_similar_question(question)

question is 'What are the top ten Consumer-to-Consumer E-commerce online?'


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

  0%|          | 0/632 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 632/632 [05:56<00:00,  1.77it/s]

no similar


In [38]:
question = qqp['test'][2]['text1']
print_similar_question(question)

question is 'Why don't people simply 'Google' instead of asking questions on Quora?'


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

  0%|          | 0/632 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 632/632 [05:56<00:00,  1.77it/s]

similar questions:
Why don't many people posting questions on Quora check Google first?
Why don't many people posting questions on Quora check Google first?
Why don't many people posting questions on Quora check Google first?
Why don't many people posting questions on Quora check Google first?
Why don't many people posting questions on Quora check Google first?


In [40]:
question = qqp['test'][3]['text1']
print_similar_question(question)

question is 'Is it safe to invest in social trade biz?'


  0%|          | 0/632 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 632/632 [05:54<00:00,  1.78it/s]

no similar


In [41]:
question = qqp['test'][4]['text1']
print_similar_question(question)

question is 'If the universe is expanding then does matter also expand?'


Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

  0%|          | 0/632 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 632/632 [05:52<00:00,  1.79it/s]

similar questions:
If universe expands and vacuum energy is created with it (with no limit),is there infinite potential energy/infinite vacuum energy that can be created?
If (theoretically) I reach the speed of light, does time apparently stop? If I surpass the speed of light, does time appear to go backwards?
If the Indian government has decided to demonetise 500 and 1000 rupee notes, why are they bringing back new 500 and 2000 Rs notes?
If the Indian government has decided to demonetise 500 and 1000 rupee notes, why are they bringing back new 500 and 2000 Rs notes?
If universe is expanding without a limit and dark and vacuum energy are created as it expands?


#### Вывод
В целом работает. 